# BİL403 — Oyuncu–takım lineage bipartite graph

Bu notebook projenin **tek çalışma yüzeyidir**:

`affiliations.csv → tekilleştirilmiş player–team edge tablosu → bipartite graph → yapısal analiz → takım ve takım-yıl analizi`

Temel kararlar:

- Oyuncu kimliği `player_node`, takım lineage kimliği `team_node` alanıdır.
- `team_name` güncel etikettir. `team_name_at_time` ve `team_name_history` yalnızca attribute olarak saklanır.
- Aynı `player_node–team_node` çiftinin bütün tarihsel/rebrand satırları **toplanır**; satır atılmaz.
- Graph yönsüz ve bipartite'tır. Ana edge weight `games_total` değeridir.
- Yapısal ölçüler ağırlıksız graph üzerinde, strength ölçüleri `games_total` ile hesaplanır.
- Başarı ve kadro sürekliliği graph metriğine indirgenmez; takım ve takım-yıl tablolarından hesaplanır.

## 1. Veriyi yükle ve sözleşmeyi doğrula

In [ ]:
from pathlib import Path
from itertools import chain

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import networkx as nx
from networkx.algorithms import bipartite
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "affiliations.csv").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "affiliations.csv"
raw = pd.read_csv(DATA_PATH)

REQUIRED_COLUMNS = {
    "player_node", "team_node", "player_name", "team_name",
    "team_name_at_time", "team_name_history", "group", "leagues",
    "active_years", "first_date", "last_date",
    "games_domestic", "wins_domestic", "minutes_domestic",
    "games_international", "wins_international", "minutes_international",
    "games_total", "wins_total", "minutes_total",
}
missing = REQUIRED_COLUMNS.difference(raw.columns)
if missing:
    raise ValueError(f"Eksik kolonlar: {sorted(missing)}")
if raw[["player_node", "team_node"]].isna().any().any():
    raise ValueError("player_node veya team_node boş olamaz.")

raw["first_date"] = pd.to_datetime(raw["first_date"], utc=True)
raw["last_date"] = pd.to_datetime(raw["last_date"], utc=True)

print(f"Kaynak satır: {len(raw):,}")
print(f"Oyuncu: {raw['player_node'].nunique():,}")
print(f"Takım lineage: {raw['team_node'].nunique():,}")
print(f"Birden fazla tarihsel satırı olan player–team çiftleri: "
      f"{raw.duplicated(['player_node', 'team_node'], keep=False).sum():,} satır")

## 2. Tarihsel satırları tek edge altında topla

Burada `drop_duplicates` kullanılmaz. Rebrand satırları aynı kimlik çifti altında
`groupby` ile toplanır; maç, dakika ve galibiyet değerleri kaybolmaz.

In [ ]:
NUMERIC_EDGE_COLUMNS = [
    "games_domestic", "wins_domestic", "minutes_domestic",
    "games_international", "wins_international", "minutes_international",
    "games_total", "wins_total", "minutes_total",
]

def pipe_union(values):
    parts = set()
    for value in values:
        parts.update(
            piece.strip()
            for piece in str(value).split("|")
            if piece.strip() and piece.strip().lower() != "nan"
        )
    return " | ".join(sorted(parts))

def year_union(values):
    years = set()
    for value in values:
        years.update(
            int(piece.strip())
            for piece in str(value).split("|")
            if piece.strip()
        )
    return sorted(years)

aggregation = {
    "player_name": "last",
    "team_name": "last",
    "team_name_at_time": pipe_union,
    "team_name_history": pipe_union,
    "group": pipe_union,
    "leagues": pipe_union,
    "active_years": lambda s: "|".join(map(str, year_union(s))),
    "first_date": "min",
    "last_date": "max",
}
aggregation.update({column: "sum" for column in NUMERIC_EDGE_COLUMNS})

edge_table = (
    raw.groupby(["player_node", "team_node"], as_index=False, sort=True)
       .agg(aggregation)
)
edge_table["domestic_win_rate"] = (
    edge_table["wins_domestic"]
    .div(edge_table["games_domestic"].replace(0, np.nan))
)

pair_count = edge_table.groupby(["player_node", "team_node"]).size()
assert pair_count.max() == 1

source_totals = raw[NUMERIC_EDGE_COLUMNS].sum()
edge_totals = edge_table[NUMERIC_EDGE_COLUMNS].sum()
if not np.allclose(source_totals.to_numpy(), edge_totals.to_numpy()):
    raise AssertionError("Toplulaştırma sırasında sayısal toplamlar değişti.")

print(f"Graph edge sayısı: {len(edge_table):,}")
print("Sayısal toplamlar korundu.")
edge_table.head()

## 3. Yönsüz bipartite graph'ı kur

In [ ]:
G = nx.Graph(
    name="LoL player–team lineage bipartite graph",
    edge_weight="games_total",
)

for row in edge_table.itertuples(index=False):
    G.add_node(
        row.player_node,
        node_type="player",
        bipartite=0,
        label=row.player_name,
    )
    G.add_node(
        row.team_node,
        node_type="team",
        bipartite=1,
        label=row.team_name,
        team_name=row.team_name,
        team_name_at_time=row.team_name_at_time,
        team_name_history=row.team_name_history,
        league=row.leagues,
        group=row.group,
    )
    G.add_edge(
        row.player_node,
        row.team_node,
        weight=float(row.games_total),
        games_total=float(row.games_total),
        games_domestic=float(row.games_domestic),
        games_international=float(row.games_international),
        wins_total=float(row.wins_total),
        minutes_total=float(row.minutes_total),
        first_date=row.first_date.isoformat(),
        last_date=row.last_date.isoformat(),
        active_years=row.active_years,
        team_name_at_time=row.team_name_at_time,
        team_name_history=row.team_name_history,
    )

assert not G.is_directed()
assert bipartite.is_bipartite(G)

def graph_for_group(graph, group_name):
    teams = {
        node for node, attrs in graph.nodes(data=True)
        if attrs["node_type"] == "team" and attrs["group"] == group_name
    }
    players = set(chain.from_iterable(set(graph.neighbors(team)) for team in teams))
    return graph.subgraph(teams | players).copy()

graphs = {
    "all": G,
    "major": graph_for_group(G, "major"),
    "wildcard": graph_for_group(G, "wildcard"),
}

print(f"Graph: {G.number_of_nodes():,} vertex, {G.number_of_edges():,} edge")

## 4. Graph ölçüleri

Degree, density, connected components ve bipartite clustering **ağırlıksız**
yapıyı açıklar. `games_total` toplamı ise weighted degree/strength olarak ayrıca
raporlanır.

In [ ]:
def graph_summary(graph, name):
    players = {n for n, a in graph.nodes(data=True) if a["node_type"] == "player"}
    teams = set(graph) - players
    components = list(nx.connected_components(graph))
    return {
        "graph": name,
        "players": len(players),
        "teams": len(teams),
        "order": graph.number_of_nodes(),
        "size": graph.number_of_edges(),
        "bipartite_density": bipartite.density(graph, players) if players and teams else np.nan,
        "connected_components": len(components),
        "largest_component_share": (
            max(map(len, components)) / graph.number_of_nodes()
            if components else np.nan
        ),
        "mean_player_degree": np.mean([graph.degree(n) for n in players]) if players else np.nan,
        "mean_team_degree": np.mean([graph.degree(n) for n in teams]) if teams else np.nan,
        "mean_bipartite_clustering": (
            bipartite.average_clustering(graph) if graph.number_of_nodes() else np.nan
        ),
    }

graph_metrics = pd.DataFrame(
    graph_summary(graph, name) for name, graph in graphs.items()
)
display(graph_metrics.round(4))

node_strength = pd.DataFrame(
    {
        "node": node,
        "label": attrs["label"],
        "node_type": attrs["node_type"],
        "degree": G.degree(node),
        "strength_games": G.degree(node, weight="games_total"),
    }
    for node, attrs in G.nodes(data=True)
)

display(
    node_strength.sort_values("strength_games", ascending=False)
                 .groupby("node_type", group_keys=False)
                 .head(10)
)

## 5. Lig-renkli, filtrelenebilir graph görseli

In [ ]:
EDGE_WEIGHT = "games_total"
TOP_PLAYERS_TO_DRAW = 45
MIN_PLAYER_DEGREE = 2
PLAYER_LABEL_COUNT = 10

LEAGUE_COLORS = {
    "LCK": "#E74C3C", "LPL": "#F39C12", "LEC": "#3498DB",
    "LCS": "#9B59B6", "LMS": "#1ABC9C",
    "TCL": "#D35400", "CBLOL": "#27AE60", "LCL": "#7F8C8D",
    "LJL": "#E84393", "OPL": "#2C3E50",
}

def visual_subgraph(graph):
    players = [
        n for n, a in graph.nodes(data=True)
        if a["node_type"] == "player" and graph.degree(n) >= MIN_PLAYER_DEGREE
    ]
    players = sorted(
        players,
        key=lambda n: graph.degree(n, weight=EDGE_WEIGHT),
        reverse=True,
    )[:TOP_PLAYERS_TO_DRAW]
    teams = set(chain.from_iterable(set(graph.neighbors(p)) for p in players))
    return graph.subgraph(set(players) | teams).copy(), players

def draw_bipartite(graph, title, ax):
    shown, players = visual_subgraph(graph)
    if not shown:
        ax.set_title(f"{title} — gösterilecek düğüm yok")
        ax.axis("off")
        return

    pos = nx.bipartite_layout(
        shown, players, align="vertical", scale=1.0, aspect_ratio=1.8
    )
    team_nodes = [n for n in shown if shown.nodes[n]["node_type"] == "team"]
    edge_values = np.array(
        [shown[u][v][EDGE_WEIGHT] for u, v in shown.edges()], dtype=float
    )
    edge_widths = 0.35 + 2.2 * np.log1p(edge_values) / np.log1p(edge_values.max())

    nx.draw_networkx_edges(
        shown, pos, ax=ax, width=edge_widths, alpha=0.24, edge_color="#667085"
    )
    nx.draw_networkx_nodes(
        shown, pos, nodelist=players, ax=ax,
        node_size=42, node_color="#D0D5DD", edgecolors="#667085", linewidths=0.4
    )
    nx.draw_networkx_nodes(
        shown, pos, nodelist=team_nodes, ax=ax,
        node_size=135,
        node_color=[
            LEAGUE_COLORS.get(shown.nodes[n]["league"], "#111827")
            for n in team_nodes
        ],
        node_shape="s", edgecolors="white", linewidths=0.7,
    )

    labelled_players = sorted(
        players,
        key=lambda n: shown.degree(n, weight=EDGE_WEIGHT),
        reverse=True,
    )[:PLAYER_LABEL_COUNT]
    labels = {
        n: shown.nodes[n]["label"]
        for n in team_nodes + labelled_players
    }
    nx.draw_networkx_labels(shown, pos, labels=labels, font_size=7, ax=ax)
    ax.set_title(
        f"{title}: {len(players)} oyuncu, {len(team_nodes)} takım\n"
        "çizgi kalınlığı = games_total"
    )
    ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(20, 11))
draw_bipartite(graphs["major"], "Büyük ligler", axes[0])
draw_bipartite(graphs["wildcard"], "Wildcard ligler", axes[1])

legend = [
    Line2D([0], [0], marker="s", color="w", label=league,
           markerfacecolor=color, markersize=9)
    for league, color in LEAGUE_COLORS.items()
]
fig.legend(handles=legend, loc="lower center", ncol=10, frameon=False)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

## 6. Takım ve takım-yıl tabloları

- Tahmini domestic takım maçı = oyuncuların `games_domestic` toplamı / 5.
- Başarı değişkeni = domestic takım galibiyeti / aktif yıl sayısı.
- Takımlar yalnızca **kendi ligleri içinde** yüzdelik sıraya dönüştürülür.
- Uluslararası katılım öncelikle `games_international > 0` ikili göstergesidir.
  Veri doğrudan Worlds ve MSI ayrımı sağlamaz.
- `active_years` yıllık oyuncu setlerine açılır. Rebrand satırlarında aynı
  `player_node–team_node–year` üyeliği set içinde doğal olarak tekilleşir.
- Yedekler de kayıtta bulunduğu için ölçünün adı **recorded roster continuity**'dir;
  bu bir “core five retention” ölçüsü değildir.

In [ ]:
team_base = (
    raw.groupby("team_node", as_index=False, sort=True)
       .agg(
           team_name=("team_name", "last"),
           team_name_at_time=("team_name_at_time", pipe_union),
           team_name_history=("team_name_history", pipe_union),
           league=("leagues", pipe_union),
           group=("group", pipe_union),
           games_domestic_player_sum=("games_domestic", "sum"),
           wins_domestic_player_sum=("wins_domestic", "sum"),
           minutes_domestic_player_sum=("minutes_domestic", "sum"),
           games_international_player_sum=("games_international", "sum"),
           wins_international_player_sum=("wins_international", "sum"),
           games_total_player_sum=("games_total", "sum"),
           active_year_list=("active_years", year_union),
       )
)

team_table = team_base.copy()
team_table["active_years"] = team_table["active_year_list"].map(
    lambda years: "|".join(map(str, years))
)
team_table["active_year_count"] = team_table["active_year_list"].map(len)
team_table["domestic_games"] = team_table["games_domestic_player_sum"] / 5
team_table["domestic_wins"] = team_table["wins_domestic_player_sum"] / 5
team_table["domestic_games_per_active_year"] = (
    team_table["domestic_games"] / team_table["active_year_count"]
)
team_table["domestic_wins_per_active_year"] = (
    team_table["domestic_wins"] / team_table["active_year_count"]
)
team_table["domestic_win_rate"] = (
    team_table["domestic_wins"]
    .div(team_table["domestic_games"].replace(0, np.nan))
)
team_table["international_participation"] = (
    team_table["games_international_player_sum"] > 0
).astype(int)

# Başarı kıyaslaması yalnızca aynı lig içinde ve aktif yıl başına yapılır.
team_table["success_percentile_in_league"] = (
    team_table.groupby("league")["domestic_wins_per_active_year"]
              .rank(method="average", pct=True)
)

membership_rows = []
for row in raw[["player_node", "team_node", "active_years"]].itertuples(index=False):
    for year in year_union([row.active_years]):
        membership_rows.append(
            {"team_node": row.team_node, "year": year, "player_node": row.player_node}
        )

memberships = pd.DataFrame(membership_rows)
roster_sets = (
    memberships.groupby(["team_node", "year"])["player_node"]
               .agg(lambda values: set(values))
               .reset_index(name="players")
)

continuity_rows = []
for team_node, history in roster_sets.groupby("team_node", sort=True):
    history = history.sort_values("year")
    previous_year = None
    previous_players = None
    for row in history.itertuples(index=False):
        current_players = row.players
        is_consecutive = (
            previous_year is not None and row.year == previous_year + 1
        )
        retained = (
            len(previous_players & current_players) if is_consecutive else np.nan
        )
        union_size = (
            len(previous_players | current_players) if is_consecutive else np.nan
        )
        continuity_rows.append(
            {
                "team_node": team_node,
                "year": row.year,
                "previous_year": previous_year if is_consecutive else np.nan,
                "recorded_roster_size": len(current_players),
                "retained_recorded_players": retained,
                "recorded_roster_continuity": (
                    retained / union_size if is_consecutive and union_size else np.nan
                ),
                "retained_share_of_previous_roster": (
                    retained / len(previous_players)
                    if is_consecutive and previous_players else np.nan
                ),
                "player_nodes": "|".join(sorted(current_players)),
            }
        )
        previous_year = row.year
        previous_players = current_players

team_year_table = pd.DataFrame(continuity_rows).merge(
    team_table[["team_node", "team_name", "league", "group"]],
    on="team_node",
    how="left",
)

retention_summary = (
    team_year_table.groupby("team_node", as_index=False)
                   .agg(
                       measured_transitions=(
                           "recorded_roster_continuity", "count"
                       ),
                       mean_recorded_roster_continuity=(
                           "recorded_roster_continuity", "mean"
                       ),
                       mean_retained_share_previous=(
                           "retained_share_of_previous_roster", "mean"
                       ),
                   )
)
team_table = team_table.merge(retention_summary, on="team_node", how="left")
team_table = team_table.drop(columns=["active_year_list"])

display(
    team_table.sort_values(
        ["league", "success_percentile_in_league"],
        ascending=[True, False],
    ).head(20)
)
display(team_year_table.head(20))

## 7. Başarı–kadro sürekliliği ilişkisi

Bu bölüm **ilişki/korelasyon** gösterir. Sonuçlar “başarı kadro korumaya neden olur”
veya tersi biçimde yorumlanamaz. Takvim, bütçe, oyuncu pazarı ve lig formatı gibi
başka değişkenler burada kontrol edilmemektedir.

In [ ]:
analysis_table = team_table[
    team_table["measured_transitions"].fillna(0).gt(0)
    & team_table["success_percentile_in_league"].notna()
].copy()

league_rows = []
for league, frame in analysis_table.groupby("league"):
    x = frame["success_percentile_in_league"]
    y = frame["mean_recorded_roster_continuity"]
    league_rows.append(
        {
            "league": league,
            "teams": len(frame),
            "spearman_success_vs_continuity": (
                x.rank().corr(y.rank()) if len(frame) >= 3 else np.nan
            ),
        }
    )
league_relationship = pd.DataFrame(league_rows)

international_comparison = (
    analysis_table.groupby("international_participation")
                  .agg(
                      teams=("team_node", "count"),
                      mean_recorded_roster_continuity=(
                          "mean_recorded_roster_continuity", "mean"
                      ),
                      median_recorded_roster_continuity=(
                          "mean_recorded_roster_continuity", "median"
                      ),
                  )
                  .reset_index()
)

display(league_relationship.round(3))
display(international_comparison.round(3))

fig, ax = plt.subplots(figsize=(10, 7))
for league, frame in analysis_table.groupby("league"):
    ax.scatter(
        frame["success_percentile_in_league"],
        frame["mean_recorded_roster_continuity"],
        s=38 + 18 * frame["international_participation"],
        alpha=0.72,
        color=LEAGUE_COLORS.get(league, "#111827"),
        label=league,
    )
ax.set(
    xlabel="Aktif yıl başına domestic galibiyet — lig içi yüzdelik",
    ylabel="Ortalama recorded roster continuity (Jaccard)",
    title="Lig içinde normalize edilmiş başarı ile kadro sürekliliği ilişkisi",
    xlim=(0, 1.02),
    ylim=(0, 1.02),
)
ax.grid(alpha=0.2)
ax.legend(ncol=2, frameon=False)
plt.show()

## 8. İsteğe bağlı dışa aktarma

In [ ]:
EXPORT_FILES = False

if EXPORT_FILES:
    export_dir = ROOT / "outputs" / "tek_notebook"
    export_dir.mkdir(parents=True, exist_ok=True)
    edge_table.to_csv(export_dir / "player_team_edges.csv", index=False)
    graph_metrics.to_csv(export_dir / "graph_summary.csv", index=False)
    team_table.to_csv(export_dir / "team_summary.csv", index=False)
    team_year_table.to_csv(export_dir / "team_year_roster_continuity.csv", index=False)
    league_relationship.to_csv(
        export_dir / "league_success_continuity_relationship.csv", index=False
    )
    nx.write_graphml(G, export_dir / "player_team_lineage.graphml")
    print("Yazıldı:", export_dir)
else:
    print("Dosya yazılmadı. Dışa aktarmak için EXPORT_FILES = True yap.")

## Yorum sınırları

- Edge varlığı oyuncunun takım lineage’ında kayıtlı en az bir maç oynadığını gösterir.
- Edge ağırlığı `games_total` değeridir; dakika ve galibiyet edge attribute'larında korunur.
- `games_international > 0` uluslararası katılımı gösterir, fakat Worlds/MSI ayrımı yapmaz.
- Recorded roster continuity, veri içinde görünen bütün oyuncuları (kısa süreli yedekler dahil) kapsar.
- Başarı–süreklilik sonuçları nedensellik değil, gözlemsel ilişkidir.